# Chapter 5 — Dimensions Do Not Mean What You Think

**Book alignment:** Embeddings From First Principles, Chapter 5

**Question this notebook isolates:** Does an arbitrary orthogonal rotation of an embedding
matrix leave *every* retrieval-relevant quantity exactly unchanged while scrambling *every*
per-axis statistic — and does RELATE's measured "similarity origin" really range from ~0.06
to ~0.45 across models?

The rotation proof is self-contained NumPy on a synthetic matrix. The anisotropy numbers
are read from the committed Wave 1 / Wave 2 artifacts.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"
aniso = json.loads((EXP / "wave1" / "artifacts" / "anisotropy.json").read_text())
rot = json.loads((EXP / "wave2" / "artifacts" / "rotation-invariance.json").read_text())

## 1. Rotate the space: retrieval is exactly invariant, axes are scrambled

Build a synthetic anisotropic embedding matrix, a random orthogonal `R`, and compare
`X` against `X @ R`.

In [ ]:
n, d = 400, 64
X = rng.standard_normal((n, d))
X[:, 0] *= 6.0                                   # a dominant 'nuisance' direction
X = X / np.linalg.norm(X, axis=1, keepdims=True)

Q, _ = np.linalg.qr(rng.standard_normal((d, d)))  # random orthogonal matrix
Xr = X @ Q

cos_before = X @ X.T
cos_after = Xr @ Xr.T
knn_before = np.argsort(-cos_before, axis=1)[:, 1:11]
knn_after = np.argsort(-cos_after, axis=1)[:, 1:11]

print("max |cos change| under rotation :", np.abs(cos_before - cos_after).max())
print("10-NN lists identical           :", bool((knn_before == knn_after).all()))
print("highest-variance axis, before   : dim", int(X.var(0).argmax()))
print("highest-variance axis, after    : dim", int(Xr.var(0).argmax()))

assert np.abs(cos_before - cos_after).max() < 1e-10
assert (knn_before == knn_after).all()
assert X.var(0).argmax() != Xr.var(0).argmax()
print("\ncosine / distance / neighbourhoods: exactly invariant. 'dimension k means X': not.")

## 2. The same result, measured on RELATE

Wave 2 applied three random rotations to real embeddings and re-ran retrieval.

In [ ]:
print("nDCG@10 identity :", rot["ndcg10_identity"])
print("nDCG@10 rotated  :", rot["ndcg10_rotated"])
print("max abs change   :", rot["max_abs_change"])
print("top-axis vs text-length corr : before", rot["top_axis_length_corr_before"],
      "-> after", rot["top_axis_length_corr_after_rotation"])

assert rot["max_abs_change"] < 1e-6
assert rot["ndcg10_rotated"] == [rot["ndcg10_identity"]] * 3
# a length signal on the top axis before rotation; scattered across the basis after
assert abs(rot["top_axis_length_corr_before"]) > abs(rot["top_axis_length_corr_after_rotation"]) - 0.1
print("\nretrieval is rotation-invariant to numerical precision; axis interpretation is not")

## 3. Anisotropy: the raw cosine scale has a non-zero, model-specific origin

For an anisotropic model, two *unrelated* sentences already score 0.3–0.45 from the shared
cone. A "relevant" 0.6 is then barely above the floor.

In [ ]:
rows = [(m, b["mean_random_cosine"], b["pct_pairs_above_0"], b["effective_rank"], b["embedding_dim"])
        for m, b in aniso["models"].items()]
for m, mrc, pct, er, dim in sorted(rows, key=lambda r: r[1]):
    print(f"  {m:14} origin={mrc:.3f}   pairs>0={pct:.0%}   eff.rank={er:.0f} / {dim}")

origins = [r[1] for r in rows]
print(f"\nsimilarity origin ranges {min(origins):.2f} .. {max(origins):.2f} across five models")
assert min(origins) < 0.10 < 0.30 < max(origins)
# the three anisotropic models have EVERY random pair at positive cosine
assert sum(1 for _, _, pct, _, _ in rows if pct == 1.0) == 3
# effective rank is a few hundred regardless of nominal dimension
assert all(er < dim for _, _, _, er, dim in rows)
print("interpreting an absolute cosine without subtracting this origin reads a number the space lacks")

## What we earned

Retrieval-relevant quantities — inner products, distances, neighbourhoods, retrieval
metrics — are *exactly* invariant to an orthogonal change of basis; every per-axis
statistic is not. So "dimension 173 means formality" cannot survive a rotation, and almost
no such claim does. And the raw cosine scale has a measured, model-specific origin (0.06 to
0.45 on RELATE) that you must subtract before an absolute similarity means anything.

**Notebook 06 / Chapter 6** looks directly at local structure — nearest neighbours, density,
hubs — and finds it lumpier than the global picture suggests.